In [ ]:

import requests
import pandas as pd
import time
import random
from tqdm import tqdm

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "application/json"
}

BASE_URL = "https://bama.ir/cad/api/search"

In [ ]:
def get_page(page):

    url = f"{BASE_URL}?pageIndex={page}&pageSize=30"

    r = requests.get(url, headers=headers, timeout=20)

    r.raise_for_status()

    return r.json()

In [ ]:
def parse_ad(ad):

    detail = ad.get("detail", {})
    specs = ad.get("specs", {})
    price = ad.get("price", {})

    mileage = detail.get("mileage", "0")

    if mileage:
        mileage = (
            mileage.replace(",", "")
                    .replace(" km", "")
                    .replace("km", "")
                    .strip()
        )
        try:
            mileage = int(mileage)
        except:
            mileage = 0
    else:
        mileage = 0

    engine = specs.get("volume")

    if engine:
        engine = engine.replace(" لیتر", "").strip()

        try:
            engine = float(engine)
        except:
            engine = None
    else:
        engine = None

    price_value = str(price.get("price", "0"))

    price_value = price_value.replace(",", "").strip()

    try:
        price_value = int(price_value)
    except:
        price_value = 0

    return {

        "brand": detail.get("brand_fa"),

        "model": detail.get("title"),

        "year": int(detail.get("year")) if detail.get("year") else None,

        "mileage_km": mileage,

        "city": detail.get("location"),

        "price": price_value,

        "price_type": price.get("type"),

        "transmission": detail.get("transmission"),

        "body_color": detail.get("body_color"),

        "inside_color": detail.get("inside_color"),

        "body_status": detail.get("body_status"),

        "engine_liter": engine,

        "link": "https://bama.ir" + detail.get("url", "")

    }

In [29]:
j = get_page(0)

ads = [x for x in j["data"]["ads"] if x["type"] == "ad"]

In [ ]:

cars = [parse_ad(x) for x in ads]

df = pd.DataFrame(cars)

df.head()

,brand,model,year,mileage_km,city,price,price_type,transmission,body_color,inside_color,body_status,engine_liter,link
0,تویوتا,تویوتا، کمری هیبرید,2026,0,تهران / قصر,0,negotiable,اتوماتیک,سفید,نارنجی,بدون رنگ,2.5,https://bama.ir/car/detail-pnytkera-toyota-cam...
1,بی وای دی,بی وای دی، سی لاین 06 DMi,2025,0,تهران / قصر,0,negotiable,اتوماتیک,کرم,مشکی,بدون رنگ,1.5,https://bama.ir/car/detail-uocg65ca-byd-sealio...
2,دیگنیتی,دیگنیتی، پرایم,1404,0,تهران / جمهوری,2300000000,lumpsum,اتوماتیک,مشکی,مشکی,بدون رنگ,1.5,https://bama.ir/car/detail-ditz5up2-dignity-pr...
3,ب ام و,ب ام و، X4,2016,114000,تهران / زعفرانیه,16300000000,lumpsum,اتوماتیک,مشکی,موکا,بدون رنگ,2.0,https://bama.ir/car/detail-vaudlnat-bmw-x4-28-...
4,هیوندای,هیوندای، سانتافه,2016,90000,تهران / یوسف‌آباد,0,negotiable,اتوماتیک,سفید صدفی,مشکی,بدون رنگ,2.4,https://bama.ir/car/detail-evz065bs-hyundai-sa...


In [ ]:
import os

MAX_RECORDS = 50000
SAVE_EVERY = 100


if os.path.exists("bama_backup.csv"):

    df_old = pd.read_csv("bama_backup.csv")

    all_cars = df_old.to_dict("records")

    print(f"Loaded {len(all_cars)} previous records.")

else:

    all_cars = []


if os.path.exists("checkpoint.txt"):

    with open("checkpoint.txt","r") as f:

        page = int(f.read())

    print(f"Continue from page {page}")

else:

    page = 0


while len(all_cars) < MAX_RECORDS:

    print(f"\nPage {page}")

    try:

        j = get_page(page)

    except Exception as e:

        print(e)

        break

    ads = [x for x in j["data"]["ads"] if x["type"]=="ad"]

    if len(ads)==0:

        print("Finished.")

        break

    for ad in ads:

        all_cars.append(parse_ad(ad))

        if len(all_cars)%SAVE_EVERY==0:

            pd.DataFrame(all_cars).drop_duplicates(subset="link").to_csv(
                "bama_backup.csv",
                index=False,
                encoding="utf-8-sig"
            )

            with open("checkpoint.txt","w") as f:

                f.write(str(page))

            print(f"Saved {len(all_cars)} cars.")

        if len(all_cars)>=MAX_RECORDS:

            break

    page += 1

    time.sleep(random.uniform(0.8,1.5))

Loaded 27112 previous records.
Continue from page 906

Page 906

Page 907
Finished.


In [ ]:
df_final = pd.DataFrame(all_cars)

df_final = df_final.drop_duplicates(subset="link")

df_final.to_csv(
    "bama_ml_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"\nFinished! {len(df_final)} unique cars.")


Finished! 27125 unique cars.
